# GraphPad Prism Data Explorer

Parse `.prism` files and plot XY data with mean ± SEM across replicates.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from prism_parser import parse_prism

PRISM_FILE = "Prism_Data.prism"
sheets = parse_prism(PRISM_FILE)
print(f"Loaded {len(sheets)} sheets: {list(sheets.keys())}")

# ---------- Color scheme configuration ----------
# Choose a matplotlib colormap for the dose columns (e.g. -12 through -6).
# Good options: "viridis", "plasma", "inferno", "magma", "cividis",
#               "coolwarm", "RdYlBu", "Spectral", "turbo"
DOSE_CMAP = "plasma"

# Fixed colors for the control / reference datasets
COLOR_BUFFER = "grey"
COLOR_FSK = "black"

# ---------- Line width ----------
LINE_WIDTH = 2  # width of all plotted lines

# ---------- Axis limits ----------
X_MIN = 0    # minutes
X_MAX = 30   # minutes


def get_color_map(dataset_names, cmap_name=DOSE_CMAP):
    """Return a dict mapping dataset name -> color.

    Numeric-looking dose labels (e.g. '-11', '-6') are spread across
    the chosen colormap.  'buffer' and '10uM Fsk' get fixed colors.
    """
    dose_names = [n for n in dataset_names if n not in ("buffer", "10uM Fsk")]
    dose_names.sort(key=lambda s: float(s))
    cmap = mpl.colormaps[cmap_name].resampled(max(len(dose_names), 1))
    colors = {}
    for i, name in enumerate(dose_names):
        colors[name] = cmap(i / max(len(dose_names) - 1, 1))
    colors["buffer"] = COLOR_BUFFER
    colors["10uM Fsk"] = COLOR_FSK
    return colors


## Plot all sheets — mean ± SEM per dataset

In [ ]:
for sheet_title, sheet in sheets.items():
    df = sheet.df
    x = df["X"].values
    y_data = df.drop(columns="X")
    dataset_names = y_data.columns.get_level_values("dataset").unique()
    colors = get_color_map(dataset_names)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax_fsk = ax.twinx()  # right y-axis for FSK

    for ds_name in dataset_names:
        ds_cols = y_data[ds_name]
        mean = ds_cols.mean(axis=1)
        sem = ds_cols.sem(axis=1)

        target_ax = ax_fsk if ds_name == "10uM Fsk" else ax
        linestyle = "--" if ds_name == "10uM Fsk" else "-"
        target_ax.plot(x, mean, label=ds_name, color=colors[ds_name], linestyle=linestyle, linewidth=LINE_WIDTH)
        target_ax.fill_between(x, mean - sem, mean + sem, alpha=0.08, color=colors[ds_name])

    ax.set_xlim(X_MIN, X_MAX)
    ax.set_xlabel(sheet.xlabel)
    ax.set_ylabel(sheet.ylabel)
    ax_fsk.set_ylabel("10uM Fsk  " + sheet.ylabel, color=COLOR_FSK)
    ax_fsk.tick_params(axis="y", labelcolor=COLOR_FSK)
    ax.set_title(sheet.graph_title)
    ax.axhline(0, color="grey", linewidth=0.5, linestyle="--")

    # Combine legends from both axes
    handles_l, labels_l = ax.get_legend_handles_labels()
    handles_r, labels_r = ax_fsk.get_legend_handles_labels()
    ax.legend(handles_l + handles_r, labels_l + labels_r, loc="best", fontsize="small")

    plt.tight_layout()

    filename = sheet.graph_title.replace(" ", "_").replace("/", "-") + ".png"
    fig.savefig(filename, dpi=150)
    print(f"Saved {filename}")

    plt.show()


## Combined Figure Plot

In [ ]:
import math

n_sheets = len(sheets)
n_cols = 2
n_rows = math.ceil(n_sheets / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(10 * n_cols, 5 * n_rows), squeeze=False)

# Hide any unused subplots
for idx in range(n_sheets, n_rows * n_cols):
    axes[idx // n_cols][idx % n_cols].set_visible(False)

for idx, (sheet_title, sheet) in enumerate(sheets.items()):
    ax = axes[idx // n_cols][idx % n_cols]
    ax_fsk = ax.twinx()

    df = sheet.df
    x = df["X"].values
    y_data = df.drop(columns="X")
    dataset_names = y_data.columns.get_level_values("dataset").unique()
    colors = get_color_map(dataset_names)

    for ds_name in dataset_names:
        ds_cols = y_data[ds_name]
        mean = ds_cols.mean(axis=1)
        sem = ds_cols.sem(axis=1)

        target_ax = ax_fsk if ds_name == "10uM Fsk" else ax
        linestyle = "--" if ds_name == "10uM Fsk" else "-"
        target_ax.plot(x, mean, label=ds_name, color=colors[ds_name], linestyle=linestyle, linewidth=LINE_WIDTH)
        target_ax.fill_between(x, mean - sem, mean + sem, alpha=0.08, color=colors[ds_name])

    ax.set_xlim(X_MIN, X_MAX)
    ax.set_xlabel(sheet.xlabel)
    ax.set_ylabel(sheet.ylabel)
    ax_fsk.set_ylabel("10uM Fsk  " + sheet.ylabel, color=COLOR_FSK)
    ax_fsk.tick_params(axis="y", labelcolor=COLOR_FSK)
    ax.set_title(sheet.graph_title)
    ax.axhline(0, color="grey", linewidth=0.5, linestyle="--")

    handles_l, labels_l = ax.get_legend_handles_labels()
    handles_r, labels_r = ax_fsk.get_legend_handles_labels()
    ax.legend(handles_l + handles_r, labels_l + labels_r, loc="best", fontsize="small")

plt.tight_layout()
plt.savefig("all_sheets.png", dpi=150)
plt.show()


## Export each sheet to .csv and .pkl

In [ ]:
import os

output_dir = "exports"
os.makedirs(output_dir, exist_ok=True)

for sheet_title, sheet in sheets.items():
    safe_name = sheet_title.replace("/", "-")
    csv_path = os.path.join(output_dir, f"{safe_name}.csv")
    pkl_path = os.path.join(output_dir, f"{safe_name}.pkl")
    sheet.df.to_csv(csv_path)
    sheet.df.to_pickle(pkl_path)
    print(f"Saved {csv_path}  |  {pkl_path}")


## Smoothed plots (Gaussian filter)

> ⚠️ **WARNING: FOR VISUAL ASSESSMENT ONLY**
> Gaussian smoothing alters the underlying data. These plots **must not** be used in publications, presentations, or any context where data integrity is required. Use the unsmoothed plots (cells above) for all reporting purposes.

In [ ]:
import warnings
warnings.warn(
    "SMOOTHED DATA — FOR VISUAL ASSESSMENT ONLY. "
    "Do NOT use these plots for publication or quantitative analysis.",
    UserWarning, stacklevel=1
)

from scipy.ndimage import gaussian_filter1d

# Sigma controls the amount of smoothing — increase for more smoothing
SMOOTH_SIGMA = 1.2

for sheet_title, sheet in sheets.items():
    df = sheet.df
    x = df["X"].values
    y_data = df.drop(columns="X")
    dataset_names = y_data.columns.get_level_values("dataset").unique()
    colors = get_color_map(dataset_names)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax_fsk = ax.twinx()

    for ds_name in dataset_names:
        ds_cols = y_data[ds_name]
        mean = gaussian_filter1d(ds_cols.mean(axis=1), sigma=SMOOTH_SIGMA)
        sem  = gaussian_filter1d(ds_cols.sem(axis=1),  sigma=SMOOTH_SIGMA)

        target_ax = ax_fsk if ds_name == "10uM Fsk" else ax
        linestyle = "--" if ds_name == "10uM Fsk" else "-"
        target_ax.plot(x, mean, label=ds_name, color=colors[ds_name], linestyle=linestyle, linewidth=LINE_WIDTH)
        target_ax.fill_between(x, mean - sem, mean + sem, alpha=0.08, color=colors[ds_name])

    ax.set_xlim(X_MIN, X_MAX)
    ax.set_xlabel(sheet.xlabel)
    ax.set_ylabel(sheet.ylabel)
    ax_fsk.set_ylabel("10uM Fsk  " + sheet.ylabel, color=COLOR_FSK)
    ax_fsk.tick_params(axis="y", labelcolor=COLOR_FSK)
    ax.set_title(sheet.graph_title + " (smoothed — visual only)")
    ax.axhline(0, color="grey", linewidth=0.5, linestyle="--")

    handles_l, labels_l = ax.get_legend_handles_labels()
    handles_r, labels_r = ax_fsk.get_legend_handles_labels()
    ax.legend(handles_l + handles_r, labels_l + labels_r, loc="best", fontsize="small")

    plt.tight_layout()
    filename2 = sheet.graph_title.replace(" ", "_").replace("/", "-") + "_smoothed.png"
    fig.savefig(filename2, dpi=150)
    print(f"Saved {filename2}")
    plt.show()
